# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 22.84 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
# YOUR CHANGES HERE
import pandas as pd
parsed_rows = [] #store in dict
for d in train_data['data']:
    title = d['title']
    for p_idx, paragraph in enumerate(d['paragraphs']):
        parsed_rows.append({
            'document_title': title,
            'paragraph_index': p_idx,
            'paragraph_context': paragraph['context']
        })
#save in df
parsed_df = pd.DataFrame(parsed_rows)
parsed_df

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...
...,...,...,...
18891,Kathmandu,53,"Institute of Medicine, the central college of ..."
18892,Kathmandu,54,Football and Cricket are the most popular spor...
18893,Kathmandu,55,The total length of roads in Nepal is recorded...
18894,Kathmandu,56,The main international airport serving Kathman...


Submit "parsed.tsv" in Gradescope.

In [15]:
parsed_df.to_csv('parsed.tsv', sep='\t', index=False, encoding='utf-8', quotechar='"')

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [16]:
# YOUR CHANGES HERE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [17]:
# YOUR CHANGES HERE
import numpy as np

#fit TF-IDF on all paragraph texts
vectorizer = TfidfVectorizer(max_features=50000, sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(parsed_df['paragraph_context'])

In [18]:
#reduce dimension via SVD
svd = TruncatedSVD(n_components=1024, random_state=42, algorithm='randomized', n_iter=2)
paragraph_vectors = svd.fit_transform(tfidf_matrix)

In [19]:
#L2-normalize vectors
paragraph_vectors = normalize(paragraph_vectors, norm='l2')

In [32]:
#save output in df
parsed_df['paragraph_vector_json'] = [
    json.dumps([round(float(x), 4) for x in vec]) for vec in paragraph_vectors
]
output_df = parsed_df[['document_title', 'paragraph_index', 'paragraph_vector_json']]
output_df

,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.2503, 0.0174, 0.0341, -0.014, -0.1, 0.0112,..."
1,University_of_Notre_Dame,1,"[0.3047, 0.06, 0.0379, -0.1055, -0.0722, 0.132..."
2,University_of_Notre_Dame,2,"[0.2532, 0.0049, 0.1363, -0.0323, -0.1438, 0.1..."
3,University_of_Notre_Dame,3,"[0.1835, -0.0186, 0.0709, -0.0305, -0.1005, 0...."
4,University_of_Notre_Dame,4,"[0.3373, 0.0621, 0.0351, -0.0888, -0.0948, 0.1..."
...,...,...,...
18891,Kathmandu,53,"[0.1965, 0.0158, 0.1485, -0.0438, -0.1401, 0.2..."
18892,Kathmandu,54,"[0.2964, 0.004, 0.1808, -0.079, -0.0768, -0.03..."
18893,Kathmandu,55,"[0.2651, 0.0405, 0.1567, 0.0269, 0.028, -0.046..."
18894,Kathmandu,56,"[0.2509, -0.0096, 0.1796, -0.0224, 0.0048, -0...."


In [33]:
#save as tsv.gz
output_df.to_csv('paragraph-vectors.tsv.gz', sep='\t', index=False, compression='gzip')

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [35]:
# YOUR CHANGES HERE
#load questions file
questions_df = pd.read_csv('questions.tsv', sep='\t')
#transform using same vectorizer and svd from Part 3
question_tfidf = vectorizer.transform(questions_df['question'])
question_vectors = svd.transform(question_tfidf)
question_vectors = normalize(question_vectors, norm='l2')

#save output in df
questions_df['question_vector_json'] = [json.dumps(vec.tolist()) for vec in question_vectors]
output_df = questions_df[['question_id', 'question_vector_json']]
output_df

,question_id,question_vector_json
0,1,"[0.2160630017367293, -0.07625626790681825, 0.0..."
1,4,"[0.1099758517030024, 0.019477146826050933, 0.0..."
2,7,"[0.11413985328200571, -0.028990468162477703, -..."
3,10,"[0.12540984385164053, -0.006726329160431841, -..."
4,13,"[0.1371259281368589, 0.07868306900206665, 0.05..."
...,...,...
95,286,"[0.1161539070527726, -0.09836258363767544, 0.0..."
96,289,"[0.07183112076198825, -0.07542765123209041, -0..."
97,292,"[0.16046193652077242, -0.006024799390450647, 0..."
98,295,"[0.15822146011714305, 0.05910737375445915, 0.0..."


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [36]:
# YOUR CHANGES HERE
output_df.to_csv('question-vectors.tsv', sep='\t', index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [37]:
# YOUR CHANGES HERE
from sklearn.neighbors import NearestNeighbors

#fit nearest neighbors on paragraph vectors
nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
nn.fit(paragraph_vectors)

#find top 5 nearest paragraphs for each question
distances, indices = nn.kneighbors(question_vectors)

#build output rows
rows = []
for q_pos, question_id in enumerate(questions_df['question_id']):
    for rank, para_idx in enumerate(indices[q_pos]):
        rows.append({
            'question_id': question_id,
            'question_rank': rank + 1,  # 1-indexed rank
            'document_title': parsed_df.iloc[para_idx]['document_title'],
            'paragraph_index': parsed_df.iloc[para_idx]['paragraph_index']
        })

#save to df
matches_df = pd.DataFrame(rows)
matches_df

,question_id,question_rank,document_title,paragraph_index
0,1,1,Tibet,10
1,1,2,Near_East,57
2,1,3,Boston,12
3,1,4,Tuvalu,49
4,1,5,Friedrich_Hayek,31
...,...,...,...,...
495,298,1,Cyprus,40
496,298,2,Cyprus,12
497,298,3,Cyprus,20
498,298,4,Cyprus,2


Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [38]:
# YOUR CHANGES HERE
matches_df.to_csv('question-matches.tsv', sep='\t', index=False)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [39]:
#get the first 5 questions sorted by question_id ascending
first_5_ids = sorted(questions_df['question_id'].unique())[:5]

worst_rows = []
for question_id in first_5_ids:
    #get the 5 matches for this section, ranked 1-5
    q_matches = matches_df[matches_df['question_id'] == question_id].sort_values('question_rank')

    #print question and all matched paragraphs for review
    question_text = questions_df[questions_df['question_id'] == question_id]['question'].values[0]
    print(f"\nQuestion ID: {question_id}")
    print(f"Question: {question_text}")
    print("Matched paragraphs:")

    for _, row in q_matches.iterrows():
        para_text = parsed_df[
            (parsed_df['document_title'] == row['document_title']) &
            (parsed_df['paragraph_index'] == row['paragraph_index'])
        ]['paragraph_context'].values[0]
        print(f"\n  Rank {row['question_rank']} | {row['document_title']} | para {row['paragraph_index']}:")
        print(f"  {para_text[:200]}...")  # print first 200 chars for readability


Question ID: 1
Question: What was the goal of the abuse of region project?
Matched paragraphs:

  Rank 1 | Tibet | para 10:
  The earliest Tibetan historical texts identify the Zhang Zhung culture as a people who migrated from the Amdo region into what is now the region of Guge in western Tibet. Zhang Zhung is considered to ...

  Rank 2 | Near_East | para 57:
  One such institution is the Centre for the Study of Ancient Documents (CSAD) founded by and located centrally at Oxford University, Great Britain. Among its many activities CSAD numbers "a long-term p...

  Rank 3 | Boston | para 12:
  By the early and mid-20th century, the city was in decline as factories became old and obsolete, and businesses moved out of the region for cheaper labor elsewhere. Boston responded by initiating vari...

  Rank 4 | Tuvalu | para 49:
  The eastern shoreline of Funafuti Lagoon was modified during World War II when the airfield (what is now Funafuti International Airport) was constructed. The cora

In [40]:
#save worst paragraphs
worst_rows = [
    {'question_id': 1,  'document_title': 'Tuvalu',         'paragraph_index': 49},
    {'question_id': 4,  'document_title': 'BeiDou_Navigation_Satellite_System', 'paragraph_index': 3},
    {'question_id': 7,  'document_title': 'Beyoncé',        'paragraph_index': 1},
    {'question_id': 10, 'document_title': 'Roman_Republic',  'paragraph_index': 48},
    {'question_id': 13, 'document_title': 'San_Diego',      'paragraph_index': 16},
]
#save in df
worst_df = pd.DataFrame(worst_rows)
worst_df

,question_id,document_title,paragraph_index
0,1,Tuvalu,49
1,4,BeiDou_Navigation_Satellite_System,3
2,7,Beyoncé,1
3,10,Roman_Republic,48
4,13,San_Diego,16


In [41]:
#save in tsv
worst_df.to_csv('worst-paragraphs.tsv', sep='\t', index=False)

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [42]:
with open('acknowledgments.txt', 'w') as f:
    f.write("When working on this project, I references the following websites:")
    f.write("https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html")
    f.write("https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html")
    f.write("https://medium.com/swlh/truncated-singular-value-decomposition-svd-using-amazon-food-reviews-891d97af5d8d")